# Experiment 1: layer-wise LLaVA vision representations

This experiment evaluates every successfully generated triplet in the AMP attack-set manifest. It compares the source $x_s$, target $x_t$, and AMP adversarial source $x_{adv}$ at every hidden state returned by the LLaVA-1.5-7B vision tower. It preserves the original Experiment-1 method from `attack_llava.py`: FP16 execution, 336×336 bicubic resize, CLIP normalization, complete token tensors (no pooling or token removal), and flattening only for cosine similarity.


In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from transformers import LlavaForConditionalGeneration

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
DTYPE = torch.float16
DEVICE = torch.device("cuda")
CACHE_VERSION = 1
FORCE_RECOMPUTE_REPRESENTATIONS = False
SHOW_IMAGES = True
SHOW_PER_SAMPLE_HEATMAPS = True


def find_repo_root():
    """Find the checkout from either the repository or notebook directory."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / ".git").exists() and (candidate / "generate_amp_perturbations.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the AMP repository root from the current working directory.")


REPO_ROOT = find_repo_root()
ATTACK_DIR = REPO_ROOT / "dataset/laion_art/attack_set"
MANIFEST_PATH = ATTACK_DIR / "manifest.csv"
ATTACK_RESULTS_PATH = ATTACK_DIR / "attack_results.csv"
CACHE_DIR = ATTACK_DIR / "representations/llava_1_5_7b"
OUTPUT_CSV = REPO_ROOT / "adversarial_mislabeling_attack/llava/exp1_layerwise_cosine.csv"
SUCCESS_STATUSES = {"completed", "skipped", "success", "successful"}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required, matching attack_llava.py.")


In [ ]:
# Load the vision tower once and match AMP's preprocessing conventions exactly.
llava_model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
vision_tower = llava_model.vision_tower.to(DEVICE).eval()
del llava_model

to_tensor = transforms.ToTensor()
preprocess = transforms.Compose(
    [
        transforms.Resize((336, 336), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.Normalize(
            (0.48145466, 0.4578275, 0.40821073),
            (0.26862954, 0.26130258, 0.27577711),
        ),
    ]
)

vision_config = vision_tower.config
EXPECTED_STATE_COUNT = vision_config.num_hidden_layers + 1
EXPECTED_STATE_SHAPE = (
    1,
    (vision_config.image_size // vision_config.patch_size) ** 2 + 1,
    vision_config.hidden_size,
)
print(
    {
        "model_type": vision_config.model_type,
        "encoder_layers": vision_config.num_hidden_layers,
        "returned_hidden_states": EXPECTED_STATE_COUNT,
        "embedding_state_included": True,
        "hidden_state_shape": EXPECTED_STATE_SHAPE,
    }
)


In [ ]:
cache_counts = {"hits": 0, "extracted": 0}


def cache_name(identifier):
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(identifier)).strip("._")
    if not value:
        raise ValueError(f"Invalid empty cache identifier: {identifier!r}")
    return f"{value}.pt"


def validate_states(states):
    if not isinstance(states, (list, tuple)) or len(states) != EXPECTED_STATE_COUNT:
        return False
    return all(
        isinstance(state, torch.Tensor)
        and tuple(state.shape) == EXPECTED_STATE_SHAPE
        and state.dtype == DTYPE
        and state.device.type == "cpu"
        for state in states
    )


def load_cached_states(cache_path):
    try:
        payload = torch.load(cache_path, map_location="cpu", weights_only=True)
        states = payload["hidden_states"]
        metadata = payload["metadata"]
        shapes = [tuple(state.shape) for state in states]
        compatible = (
            metadata.get("cache_version") == CACHE_VERSION
            and metadata.get("model_id") == MODEL_ID
            and metadata.get("hidden_state_count") == len(states)
            and [tuple(shape) for shape in metadata.get("tensor_shapes", [])] == shapes
            and validate_states(states)
        )
        if not compatible:
            raise ValueError("incompatible cache metadata or tensors")
        cache_counts["hits"] += 1
        return tuple(states)
    except Exception as error:
        warnings.warn(f"Ignoring invalid representation cache {cache_path}: {error}")
        return None


def extract_hidden_states(image_path):
    """Return every full FP16 vision hidden-state token tensor on CPU."""
    with Image.open(image_path) as image:
        image_tensor = to_tensor(image.convert("RGB")).to(DEVICE, DTYPE)
    pixel_values = preprocess(image_tensor).unsqueeze(0)
    with torch.inference_mode():
        outputs = vision_tower(pixel_values, output_hidden_states=True)
    states = tuple(state.detach().to(device="cpu", dtype=DTYPE) for state in outputs.hidden_states)
    if not validate_states(states):
        raise ValueError(
            f"Unexpected hidden states for {image_path}: "
            f"count={len(states)}, shapes={[tuple(state.shape) for state in states]}"
        )
    return states


def get_hidden_states(image_path, cache_path):
    if cache_path.is_file() and not FORCE_RECOMPUTE_REPRESENTATIONS:
        cached = load_cached_states(cache_path)
        if cached is not None:
            return cached
    states = extract_hidden_states(image_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "metadata": {
            "cache_version": CACHE_VERSION,
            "model_id": MODEL_ID,
            "hidden_state_count": len(states),
            "tensor_shapes": [tuple(state.shape) for state in states],
            "dtype": str(DTYPE),
        },
        "hidden_states": states,
    }
    torch.save(payload, cache_path)
    cache_counts["extracted"] += 1
    return states


def flattened_cosine(left, right):
    """Flatten only for cosine and retain the original float32 calculation."""
    return F.cosine_similarity(
        left.float().reshape(1, -1), right.float().reshape(1, -1)
    ).item()


def analyze_sample(source_path, target_path, adversarial_path, sample_id, source_image_id, target_image_id):
    """Extract/cache one triplet and return one result row per hidden state."""
    states = {
        "source": get_hidden_states(
            source_path, CACHE_DIR / "clean" / cache_name(source_image_id)
        ),
        "target": get_hidden_states(
            target_path, CACHE_DIR / "clean" / cache_name(target_image_id)
        ),
        "adversarial": get_hidden_states(
            adversarial_path, CACHE_DIR / "adv" / cache_name(sample_id)
        ),
    }
    counts = {name: len(value) for name, value in states.items()}
    shapes = {name: [tuple(tensor.shape) for tensor in value] for name, value in states.items()}
    if len(set(counts.values())) != 1 or not (
        shapes["source"] == shapes["target"] == shapes["adversarial"]
    ):
        raise ValueError(f"Representation mismatch: counts={counts}, shapes={shapes}")

    rows = []
    for layer, (source, target, adversarial) in enumerate(
        zip(states["source"], states["target"], states["adversarial"])
    ):
        R = flattened_cosine(adversarial, source)
        T = flattened_cosine(adversarial, target)
        B = flattened_cosine(source, target)
        rows.append(
            {
                "layer": layer,
                "R": R,
                "T": T,
                "B": B,
                "G": T - B,
                "tensor_shape": str(tuple(source.shape)),
            }
        )
    del states
    return rows


In [ ]:
def layer_labels(layers):
    return ["Emb." if int(layer) == 0 else str(int(layer)) for layer in layers]


def show_triplet(source_path, adversarial_path, target_path, sample):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    labels = [
        f"Source: {sample.source_concept}",
        f"Adversarial source\n{sample.sample_id}",
        f"Target: {sample.target_concept}",
    ]
    for axis, path, label in zip(axes, [source_path, adversarial_path, target_path], labels):
        with Image.open(path) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(label)
        axis.axis("off")
    fig.suptitle(f"AMP sample {sample.sample_id}: {sample.source_concept} → {sample.target_concept}")
    fig.tight_layout()
    plt.show()


def draw_heatmap(frame, title):
    metrics = ["R", "T", "B", "G"]
    ordered = frame.sort_values("layer")
    values = ordered[metrics].to_numpy().T
    bound = max(1.0, float(np.nanmax(np.abs(values))))
    fig, axis = plt.subplots(figsize=(14, 3.5))
    image = axis.imshow(values, aspect="auto", cmap="coolwarm", vmin=-bound, vmax=bound)
    axis.set_xticks(range(len(ordered)), labels=layer_labels(ordered["layer"]), rotation=90)
    axis.set_yticks(range(len(metrics)), labels=metrics)
    axis.set_xlabel("Hidden-state index (Emb. is the embedding state)")
    axis.set_title(title)
    fig.colorbar(image, ax=axis, label="Cosine similarity / target gain")
    fig.tight_layout()
    plt.show()


In [ ]:
# Join in manifest order; paths in both CSVs are repository-root-relative.
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
attack_results = pd.read_csv(ATTACK_RESULTS_PATH, dtype=str)
required_manifest = {
    "sample_id", "pair_id", "source_path", "target_path", "source_image_id",
    "target_image_id", "source_concept", "target_concept",
}
required_results = {"sample_id", "adv_path", "status"}
for path, frame, required in [
    (MANIFEST_PATH, manifest, required_manifest),
    (ATTACK_RESULTS_PATH, attack_results, required_results),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {sorted(missing)}")
    if frame["sample_id"].duplicated().any():
        raise ValueError(f"Duplicate sample IDs in {path}")

samples = manifest.merge(
    attack_results[["sample_id", "adv_path", "status"]],
    on="sample_id",
    how="left",
    sort=False,
    validate="one_to_one",
)

provenance_columns = [
    "sample_id", "pair_id", "source_image_id", "target_image_id",
    "source_concept", "target_concept",
]
all_rows = []
skipped = []
for sample in samples.itertuples(index=False):
    status = str(sample.status).strip().lower()
    if status not in SUCCESS_STATUSES:
        skipped.append((sample.sample_id, f"attack status={sample.status!r}"))
        warnings.warn(f"Skipping {sample.sample_id}: attack status={sample.status!r}")
        continue

    source_path = REPO_ROOT / sample.source_path
    target_path = REPO_ROOT / sample.target_path
    adversarial_path = REPO_ROOT / sample.adv_path
    try:
        if SHOW_IMAGES:
            show_triplet(source_path, adversarial_path, target_path, sample)
        sample_rows = analyze_sample(
            source_path,
            target_path,
            adversarial_path,
            sample.sample_id,
            sample.source_image_id,
            sample.target_image_id,
        )
        provenance = {column: getattr(sample, column) for column in provenance_columns}
        for row in sample_rows:
            all_rows.append({**provenance, **row})
        if SHOW_PER_SAMPLE_HEATMAPS:
            draw_heatmap(
                pd.DataFrame(sample_rows),
                f"AMP Experiment 1 — {sample.sample_id}: {sample.source_concept} → {sample.target_concept}",
            )
    except Exception as error:
        skipped.append((sample.sample_id, f"{type(error).__name__}: {error}"))
        warnings.warn(f"Skipping {sample.sample_id}: {type(error).__name__}: {error}")

columns = provenance_columns + ["layer", "R", "T", "B", "G", "tensor_shape"]
results = pd.DataFrame(all_rows, columns=columns)
if not results.empty:
    results = results.sort_values(["sample_id", "layer"], kind="stable")
    # Restore deterministic manifest order rather than lexical sample-ID order.
    sample_order = {sample_id: index for index, sample_id in enumerate(manifest["sample_id"])}
    results["_sample_order"] = results["sample_id"].map(sample_order)
    results = results.sort_values(["_sample_order", "layer"], kind="stable").drop(columns="_sample_order")
results.to_csv(OUTPUT_CSV, index=False)  # Replace, never append.

analyzed_count = results["sample_id"].nunique() if not results.empty else 0
print(
    "Summary: "
    f"manifest={len(manifest)}, analyzed={analyzed_count}, skipped/failed={len(skipped)}, "
    f"cache_hits={cache_counts['hits']}, newly_extracted={cache_counts['extracted']}"
)
if skipped:
    print("Skipped samples:", "; ".join(f"{sample_id} ({reason})" for sample_id, reason in skipped))
if analyzed_count == 0:
    raise RuntimeError("No valid AMP attack samples remain; aggregate heatmap was not produced.")
results


In [ ]:
# Always compute the requested aggregate from the dataframe saved above.
saved_results = pd.read_csv(OUTPUT_CSV, dtype={"sample_id": str})
aggregate = (
    saved_results.groupby("layer", sort=True, as_index=False)[["R", "T", "B", "G"]]
    .mean()
    .sort_values("layer")
)
draw_heatmap(
    aggregate,
    f"AMP Experiment 1 aggregate — mean across {analyzed_count} analyzed AMP samples",
)
aggregate
